# Engineering Assignment E2-I: Tensile Test Analysis

**Engineering 10 · Module 2 · Fall 2026 · George Witmer**

**Level:** Intermediate

---

> **About the three levels of E2.** This assignment exists in three independent versions — Novice (E2-N), Intermediate (E2-I), and Mastery (E2-M). Each is a different problem of similar conceptual difficulty; the levels differ mainly in scope and how long they take. You may submit one, two, or all three — each is graded on its own. **You do not need to complete a lower level before attempting a higher one.**

---

## Scenario

You are a materials engineer in a testing lab. Three nominally identical **steel coupons** (small bars of metal) have just been pulled in a tensile-testing machine, which applied an increasing axial force and recorded the resulting elongation at each step. Your job is to convert the raw force–elongation data into the engineering quantities **stress** and **strain**, compute **Young's modulus** for each specimen, identify which one was the stiffest, and produce a one-page report.

## Learning Objectives

After completing this level, you will be able to:
1. Store tabular engineering data as a two-dimensional list (a list of rows, each row a small list).
2. Use a list comprehension to transform raw measurements into derived quantities.
3. Walk parallel lists in lockstep using `zip()` inside a `for` loop.
4. Define and call a small function that takes physical parameters and returns a derived quantity.
5. Use f-strings with field widths and decimal-place specifiers to produce a properly aligned numeric table.

## Prerequisites

You should be comfortable with the following topics from Module 1 and this module:
- `lists-basics`, `lists-slicing` — indexing and slicing
- `lists-2d` — lists of lists, `grid[r][c]` indexing
- `list-comprehensions` — `[f(x) for x in xs]`, including with two-variable unpacking from a 2D row
- `for-loops` — `for`, `range`, `enumerate`, `zip`
- `functions-defining` — `def`, parameters, `return` (Module 1)
- `fstrings` — field widths and decimal-place specifiers, e.g. `{x:8.2f}`

*If any of these feel shaky, review the topic notes before starting — this assignment assumes them.*

## Background

A **tensile test** is the most common mechanical test in engineering. A small bar of material — the **coupon** or **specimen** — is gripped at both ends in a testing machine, and the machine pulls the ends apart with a steadily increasing axial **load** $F$ (a force, in newtons) while measuring the resulting **extension** $\Delta L$ (an elongation, in millimeters). From these raw measurements you compute two derived quantities that are independent of the specimen's size, and therefore characterize the *material* itself rather than the particular piece you happened to test:

> **Stress** $\sigma$ is force per unit cross-sectional area:
> $$\sigma = \frac{F}{A}$$
> with $F$ in newtons and $A$ in square meters, giving $\sigma$ in pascals (Pa). Because the numbers in Pa are awkwardly large for engineering use, results are almost always reported in **megapascals (MPa)**, where $1\,\text{MPa} = 10^6\,\text{Pa}$.

> **Strain** $\varepsilon$ is the fractional change in length relative to the original gauge length $L_0$:
> $$\varepsilon = \frac{\Delta L}{L_0}$$
> with $\Delta L$ and $L_0$ in the same units, so $\varepsilon$ is **unitless** (a pure number, often very small — typical elastic strains in metals are around $10^{-3}$ or smaller).

For small loads, every common engineering metal behaves linearly: stress is proportional to strain. The constant of proportionality is **Young's modulus** $E$, also called the *modulus of elasticity*:

> $$\sigma = E\,\varepsilon \qquad \Longleftrightarrow \qquad E = \frac{\sigma}{\varepsilon}$$

$E$ has units of pressure (Pa, MPa, or GPa), and like the materials themselves it is a characteristic number: structural steel is around 200 GPa, aluminum about 70 GPa, copper around 110 GPa, concrete roughly 25 GPa. **All the data in this assignment lies in the elastic (linear) region**, so you can compute $E$ directly from any (stress, strain) pair as $\sigma / \varepsilon$ — you do not need to fit a line.

**For further reading:**
- [NIST: Tensile testing of metals (ASTM E8/E8M overview)](https://www.nist.gov/) — search for *tensile testing*.
- *Mechanics of Materials* (Hibbeler, Beer & Johnston, or any introductory text), chapter on axial loading and Hooke's law.

*You do not need to read these to complete the assignment — they are here if you want to go deeper.*

## Problem Statement

Three steel coupons, all machined to the same nominal geometry, were pulled in tension. Each coupon's data is six (load, extension) pairs in the linear-elastic region. Convert the raw data into stress and strain, compute Young's modulus for each specimen, identify the stiffest one, and print a report.

Use **this exact dataset** at the top of your notebook:

```python
AREA_M2  = 100.0e-6   # cross-sectional area, m^2  (= 100 mm^2)
GAUGE_MM = 50.0       # gauge length, mm

# Each specimen is a list of [load_kN, extension_mm] rows.
specimens = [
    # Specimen A
    [[0.0, 0.000], [5.0, 0.012], [10.0, 0.024], [15.0, 0.036], [20.0, 0.048], [25.0, 0.060]],
    # Specimen B
    [[0.0, 0.000], [5.0, 0.013], [10.0, 0.026], [15.0, 0.039], [20.0, 0.052], [25.0, 0.065]],
    # Specimen C
    [[0.0, 0.000], [5.0, 0.014], [10.0, 0.028], [15.0, 0.042], [20.0, 0.056], [25.0, 0.070]],
]
specimen_names = ["A", "B", "C"]
```

Each row inside a specimen is `[load_in_kN, extension_in_mm]`. Note the **unit gotcha**: load is in kilonewtons (kN), not newtons, and extension is in millimeters. You must convert before computing stress in Pa.

### Requirements

Your code must:
- Copy the dataset above into your notebook exactly as shown.
- Define a small function — for example, `stress_strain(load_kN, ext_mm)` — that takes one (load, extension) pair and returns a list (or tuple) `[stress_MPa, strain]`. The function should use the module-level `AREA_M2` and `GAUGE_MM` constants and handle the unit conversions internally.
- For **each specimen**, use a **list comprehension** to transform its list of `[load_kN, ext_mm]` rows into a list of `[stress_MPa, strain]` rows. (You should end up with a new 2D list `processed` of the same shape as `specimens`.)
- For **Specimen A only**, print a 4-column table showing, for each of the 6 rows: load (kN), extension (mm), stress (MPa), strain. Use `zip()` to walk the raw and processed rows in parallel inside a single `for` loop. Match the column widths shown in the sample output.
- Compute Young's modulus $E$ for each specimen using the **last row** of that specimen's processed data (the largest stress and largest strain — both positive, no divide-by-zero risk). Report $E$ in **gigapascals (GPa)**. Store the three moduli in a list.
- Using a `for` loop with `enumerate` or `range`, find the index of the **stiffest** specimen — the one with the largest $E$. (Do not use the built-in `max()`. Manual comparison is the point.)
- Compute the **mean modulus** across the three specimens.
- Print the per-specimen modulus block and the summary lines exactly as shown in the sample output.

### Expected Output

When you run your program with the dataset above, you should see exactly:

```
Tensile Test Report - Steel Coupons (ASTM E8 round bar geometry)
Cross-sectional area: 100.0 mm^2   Gauge length: 50.0 mm

Specimen A - stress/strain pairs (linear elastic region):
   Load (kN)    Ext (mm)    Stress (MPa)      Strain
        0.00      0.0000            0.00    0.000000
        5.00      0.0120           50.00    0.000240
       10.00      0.0240          100.00    0.000480
       15.00      0.0360          150.00    0.000720
       20.00      0.0480          200.00    0.000960
       25.00      0.0600          250.00    0.001200

Young's modulus per specimen (from peak-load point):
  Specimen A:  E =  208.3 GPa
  Specimen B:  E =  192.3 GPa
  Specimen C:  E =  178.6 GPa

Mean modulus across the three specimens: 193.1 GPa
Stiffest specimen: A (208.3 GPa)
```

*Reproduce the numbers and column alignment exactly. The expected results are around 200 GPa, which is in the right ballpark for structural steel.*

## Your Solution

Write your code in the cell below. You may add additional cells (code or markdown) as needed to organize your work or explain your thinking.

In [6]:
# Your solution here
AREA_M2  = 100.0e-6   # cross-sectional area, m^2  (= 100 mm^2)
GAUGE_MM = 50.0       # gauge length, mm

# Each specimen is a list of [load_kN, extension_mm] rows.
specimens = [
    # Specimen A
    [[0.0, 0.000], [5.0, 0.012], [10.0, 0.024], [15.0, 0.036], [20.0, 0.048], [25.0, 0.060]],
    # Specimen B
    [[0.0, 0.000], [5.0, 0.013], [10.0, 0.026], [15.0, 0.039], [20.0, 0.052], [25.0, 0.065]],
    # Specimen C
    [[0.0, 0.000], [5.0, 0.014], [10.0, 0.028], [15.0, 0.042], [20.0, 0.056], [25.0, 0.070]],
]
specimen_names = ["A", "B", "C"]

def stress_strain(load_kN, ext_mm):
    #converting
    stress_MPa = (load_kN * 1000) / (AREA_M2 * 1e6)
    strain = ext_mm / GAUGE_MM
    
    return [stress_MPa, strain]


processed = [ [stress_strain(load, ext) for load, ext in specimen] for specimen in specimens ]


print("Tensile Test Report - Steel Coupons (ASTM E8 round bar geometry)")
print(f"Cross-sectional area: 100.0 mm^2    Gauge length: {GAUGE_MM} mm\n")
print("Specimen A - stress/strain pairs (linear elastic region):")
print(f"{'Load (kN)':>9} {'Ext (mm)':>10} {'Stress (MPa)':>14} {'Strain':>10}")


#pairs raw values with corresponding converted ones and prints
for raw_row, proc_row in zip(specimens[0], processed[0]):
    load_kN, ext_mm = raw_row
    stress_MPa, strain = proc_row
    print(f"{load_kN:9.2f} {ext_mm:10.4f} {stress_MPa:14.2f} {strain:10.6f}")

print()


moduli_GPa = []
for proc_specimen in processed:
    last_stress, last_strain = proc_specimen[-1] #gets last row
    E_GPa = (last_stress / last_strain) / 1000.0
    moduli_GPa.append(E_GPa)

print("Young's modulus per specimen (from peak-load point):")
for name, E in zip(specimen_names, moduli_GPa):
    print(f"  Specimen {name}:  E = {E:5.1f} GPa")


max_index = 0
max_E = moduli_GPa[0]
#finds max "stiffest"
for i in range(1, len(moduli_GPa)):
    if moduli_GPa[i] > max_E:
        max_E = moduli_GPa[i]
        max_index = i

mean_E = sum(moduli_GPa) / len(moduli_GPa)


print(f"\nMean modulus across the three specimens: {mean_E:.1f} GPa")
print(f"Stiffest specimen: {specimen_names[max_index]} ({max_E:.1f} GPa)")

Tensile Test Report - Steel Coupons (ASTM E8 round bar geometry)
Cross-sectional area: 100.0 mm^2    Gauge length: 50.0 mm

Specimen A - stress/strain pairs (linear elastic region):
Load (kN)   Ext (mm)   Stress (MPa)     Strain
     0.00     0.0000           0.00   0.000000
     5.00     0.0120          50.00   0.000240
    10.00     0.0240         100.00   0.000480
    15.00     0.0360         150.00   0.000720
    20.00     0.0480         200.00   0.000960
    25.00     0.0600         250.00   0.001200

Young's modulus per specimen (from peak-load point):
  Specimen A:  E = 208.3 GPa
  Specimen B:  E = 192.3 GPa
  Specimen C:  E = 178.6 GPa

Mean modulus across the three specimens: 193.1 GPa
Stiffest specimen: A (208.3 GPa)


---

## Preparing for the Oral Exam

Your oral exam will follow a fixed rubric the TA works through with you. Use the four sections below to prepare — points are awarded only when you can do all four.

### 1. Files Submitted

Before the oral exam can begin, you must show the TA that your notebook has been uploaded to Course Site. Do this *before* the appointment, not during it.

### 2. Demonstration

You will run your code in front of the TA. It must produce the expected output with no errors and no warnings. Your code must actually compute each stress, each strain, and each modulus from the dataset — printing hardcoded `208.3`, `192.3`, `178.6` will fail this step. Your list comprehension and your `zip` loop must really run.

### 3. Explanation (no looking at your code)

The TA will ask you to describe, **without looking at your code or these notes**, what the problem is and how your solution approaches it. Practice this out loud before the exam. You should be able to say something like *"For each specimen I take its list of load-and-extension rows and use a list comprehension to make a matching list of stress-and-strain rows, calling my `stress_strain` function on each one. To print the table for Specimen A I zip the raw and processed lists together so I can show all four columns side by side. Then I take the last processed row from each specimen, divide stress by strain to get Young's modulus, store the three numbers in a list, and walk it with a `for` loop to find the maximum and compute the mean."*

### 4. Code Review

With your code in front of you, you will walk the TA through it. Be ready to explain:
- What each line or function does and why it is there.
- Specific decisions — for example, *the unit conversions inside `stress_strain` (kN → N, mm → unitless strain, Pa → MPa) and why each one is there*, *the syntax `[stress_strain(L, e) for L, e in rows]` — what the `for L, e in rows` is doing and why it works even though each row is a list rather than a tuple*, *why you took the modulus from the last row rather than the first (and what would have gone wrong with the first)*.
- How your code would behave on a variation the TA may propose — for example, *add a fourth specimen with different numbers and re-run; do you have to change anything else?*, *change the cross-sectional area to 80 mm² and confirm that the moduli stay the same but the stresses change*, *report each specimen's modulus as the **average** of $\sigma/\varepsilon$ across all non-zero rows instead of just the last point*. You may be asked to edit and re-run on the spot.

You will **not** have access to the internet, AI tools, or chat assistants during the oral exam. If you cannot explain your own code, you will not pass — regardless of how well it runs.

---

## How to Submit

1. Save this notebook with your work as `E2-I_LastnameFirstname.ipynb`.
2. Upload to Course Site under this assignment.
3. Request an oral exam from a TA **during office hours** (Fritz 402). Oral exams are not given during lab sections.
4. Points are awarded only after you pass the oral exam. If you do not pass on the first try, the TA will tell you what to revise and you may try again later — but it must still be before the module deadline.

**Deadline:** End of Module 2 — see the course schedule. **No extensions.**

---

## Academic Honesty

You may use the internet, including generative AI, to **help you learn**. You may not copy another student's work, and you may not submit code you cannot explain. The oral exam is where this gets checked: if you cannot explain your own code without electronic help, you will not pass — regardless of how good the code is.

See the [course academic honesty policy](https://www.lehigh.edu/~ejk0/cheating.html).